# InsightForge AI — Agent 6: Report Agent
Assembles all outputs into a formatted, downloadable PDF executive report using fpdf2.


In [ ]:
%pip install fpdf2 pandas
dbutils.library.restartPython()


In [ ]:
class InsightForgeState(TypedDict):
    """
    Shared state passed through all agents in the pipeline.
    Each agent reads what it needs and writes its output.
    No agent modifies another agent's output fields.

    Fields
    ------
    dataset_path     : path to the input CSV file
    gemini_key       : Gemini API key passed at runtime
    raw_df           : original DataFrame as uploaded
    cleaned_df       : DataFrame after cleaning agent runs
    schema_info      : column metadata detected by schema agent
    cleaning_report  : summary of all cleaning actions taken
    eda_results      : statistical analysis from EDA agent
    charts           : list of Plotly figure dicts from viz agent
    insights         : AI generated business insights text
    pdf_path         : path to the generated PDF report
    pipeline_log     : timestamped log of each agent execution
    errors           : list of error messages from any agent
    """
    dataset_path    : str
    gemini_key      : str
    raw_df          : Any
    cleaned_df      : Any
    schema_info     : dict
    cleaning_report : dict
    eda_results     : dict
    charts          : list
    insights        : str
    pdf_path        : str
    pipeline_log    : list
    errors          : list

print("✅ InsightForgeState defined")
print()
print("  State fields:")
fields = [
    ("dataset_path",     "input — path to CSV"),
    ("gemini_key",       "input — API key"),
    ("raw_df",           "Schema Agent reads this"),
    ("cleaned_df",       "Cleaning Agent writes this"),
    ("schema_info",      "Schema Agent writes this"),
    ("cleaning_report",  "Cleaning Agent writes this"),
    ("eda_results",      "EDA Agent writes this"),
    ("charts",           "Visualization Agent writes this"),
    ("insights",         "Insight Agent writes this"),
    ("pdf_path",         "Report Agent writes this"),
    ("pipeline_log",     "every agent appends to this"),
    ("errors",           "every agent appends on failure"),
]
for field, desc in fields:
    print(f"    {field:20} — {desc}")


In [ ]:
def log_event(state: InsightForgeState, agent: str, message: str) -> list:
    """
    Appends a timestamped log entry to the pipeline log.
    Called by every agent on start and completion.

    Parameters
    ----------
    state   : current pipeline state
    agent   : name of the calling agent
    message : what happened

    Returns
    -------
    list : updated pipeline log
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = f"[{timestamp}] {agent}: {message}"
    print(f"   {entry}")
    return state["pipeline_log"] + [entry]


def get_gemini_model() -> genai.GenerativeModel:
    """
    Returns a configured Gemini model instance.
    Re-reads the API key from widget each time to handle
    session restarts without needing to re-run setup cells.
    """
    key = dbutils.widgets.get("gemini_key")
    genai.configure(api_key=key)
    return genai.GenerativeModel(GEMINI_MODEL)


def safe_call_gemini(prompt: str, agent_name: str) -> str:
    """
    Wraps a Gemini API call with error handling.
    Cleans the response text to remove characters that
    fpdf2 cannot render with standard Helvetica font.

    Parameters
    ----------
    prompt     : the full prompt string to send
    agent_name : name of the calling agent for logging

    Returns
    -------
    str : cleaned response text or error message
    """
    try:
        m        = get_gemini_model()
        response = m.generate_content(prompt)
        text     = response.text

        # Remove characters unsupported by Helvetica in fpdf2
        replacements = {
            "\u2014": "-",    # em dash
            "\u2013": "-",    # en dash
            "\u2012": "-",    # figure dash
            "\u2011": "-",    # non-breaking hyphen
            "\u2010": "-",    # hyphen
            "\u2022": "-",    # bullet
            "\u2023": "-",    # triangle bullet
            "\u2043": "-",    # hyphen bullet
            "\u2018": "'",    # left single quote
            "\u2019": "'",    # right single quote
            "\u201a": "'",    # single low quote
            "\u201c": '"',    # left double quote
            "\u201d": '"',    # right double quote
            "\u201e": '"',    # double low quote
            "\u2026": "...",  # ellipsis
            "\u00a0": " ",    # non-breaking space
            "\u00b7": "-",    # middle dot
            "\u2015": "-",    # horizontal bar
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)

        # Final safety pass — replace remaining non-latin-1 chars
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    except Exception as e:
        logger.warning(
            f"Gemini call failed in {agent_name}: {str(e)[:100]}"
        )
        print(f"   ⚠️  Gemini call failed in {agent_name}: {e}")
        return f"[Gemini error in {agent_name}: {str(e)}]"


print("✅ Utility functions defined")
print("   log_event()        — timestamped pipeline logging")
print("   get_gemini_model() — safe model initialisation")
print("   safe_call_gemini() — error handled API call with font cleaning")


In [ ]:
def report_agent(state: InsightForgeState) -> dict:
    """
    Agent 6 — Report Agent
    ----------------------
    Reads  : all state fields
    Writes : state["pdf_path"]

    PDF sections:
    1. Dataset Overview
    2. Cleaning Report
    3. Statistical Summary table
    4. Key Correlations
    5. AI Insights
    6. Categorical Summary
    7. Pipeline Execution Log
    """
    agent_name = "Report Agent"
    print(f"\n{'─' * 55}")
    print(f"📄 {agent_name} starting...")

    df       = state["cleaned_df"]
    schema   = state["schema_info"]
    eda      = state["eda_results"]
    clean    = state["cleaning_report"]
    insights = state["insights"]
    pipe_log = state["pipeline_log"]
    errors   = state["errors"]
    log      = log_event(state, agent_name, "started")

    logger.info(f"{agent_name} started — assembling PDF")

    def clean_text(text) -> str:
        """Sanitises string for Helvetica font in fpdf2."""
        if not isinstance(text, str):
            text = str(text)
        replacements = {
            "\u2014":"-", "\u2013":"-", "\u2012":"-",
            "\u2011":"-", "\u2010":"-", "\u2022":"-",
            "\u2023":"-", "\u2043":"-", "\u2018":"'",
            "\u2019":"'", "\u201a":"'", "\u201c":'"',
            "\u201d":'"', "\u201e":'"', "\u2026":"...",
            "\u00a0":" ", "\u00b7":"-", "\u2015":"-",
            "\u2192":"->","\u2190":"<-","\u00d7":"x",
            "\u00f7":"/",
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    def safe_cell_value(raw_val) -> str:
        """Converts any value to a PDF-safe string."""
        try:
            if pd.isna(raw_val):
                return ""
        except (TypeError, ValueError):
            pass
        if raw_val in (float("inf"), float("-inf")):
            return "inf"
        if isinstance(raw_val, float):
            return clean_text(str(round(raw_val, 2)))
        return clean_text(str(raw_val))

    try:
        class InsightReport(FPDF):
            def header(self):
                self.set_fill_color(31, 97, 141)
                self.set_text_color(255, 255, 255)
                self.set_font("Helvetica", "B", 15)
                self.cell(
                    0, 13,
                    "InsightForge AI - Data Analysis Report",
                    align="C", fill=True,
                    new_x="LMARGIN", new_y="NEXT"
                )
                self.set_font("Helvetica", "", 8)
                self.set_text_color(120, 120, 120)
                self.cell(
                    0, 5,
                    f"Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}"
                    f"  |  {GEMINI_MODEL}"
                    f"  |  Databricks  |  LangGraph",
                    align="C",
                    new_x="LMARGIN", new_y="NEXT"
                )
                self.set_text_color(0, 0, 0)
                self.ln(3)

            def footer(self):
                self.set_y(-14)
                self.set_font("Helvetica", "I", 8)
                self.set_text_color(150, 150, 150)
                self.cell(
                    0, 10,
                    clean_text(
                        f"InsightForge AI  |  Page {self.page_no()}"
                        f"  |  LangGraph + Gemini on Databricks"
                    ),
                    align="C"
                )

            def section_header(self, title: str):
                self.set_font("Helvetica", "B", 11)
                self.set_fill_color(213, 232, 252)
                self.set_text_color(25, 80, 140)
                self.cell(
                    0, 8, clean_text(f"  {title}"),
                    fill=True, new_x="LMARGIN", new_y="NEXT"
                )
                self.set_text_color(0, 0, 0)
                self.ln(2)

            def key_value_row(self, label: str, value):
                self.set_font("Helvetica", "B", 10)
                self.cell(65, 7, clean_text(f"  {label}"), border="B")
                self.set_font("Helvetica", "", 10)
                self.cell(
                    0, 7,
                    safe_cell_value(value)[:100],
                    border="B", new_x="LMARGIN", new_y="NEXT"
                )

        pdf = InsightReport()
        pdf.add_page()

        # Section 1 — Dataset Overview
        pdf.section_header("1.  Dataset Overview")
        pdf.key_value_row("Domain",          schema.get("domain",  ""))
        pdf.key_value_row("Industry",         schema.get("industry",""))
        pdf.key_value_row("Summary",          schema.get("summary", ""))
        pdf.key_value_row("Rows",             eda["shape"][0])
        pdf.key_value_row("Columns",          eda["shape"][1])
        pdf.key_value_row("Target Variable",  schema.get("target_variable",""))
        pdf.key_value_row("Numeric Columns",  ", ".join(eda.get("numeric_cols",[])))
        pdf.key_value_row("Categorical Cols", ", ".join(eda.get("categorical_cols",[])))
        pdf.key_value_row("Remaining Nulls",  eda.get("total_nulls", 0))
        pdf.ln(4)

        # Section 2 — Cleaning Report
        pdf.section_header("2.  Data Cleaning Report")
        pdf.key_value_row("Rows Before",       clean.get("rows_before",""))
        pdf.key_value_row("Rows After",        clean.get("rows_after",""))
        pdf.key_value_row("Duplicates Removed",clean.get("duplicates_removed",0))
        pdf.key_value_row(
            "Columns Dropped",
            ", ".join(clean.get("columns_dropped",[])) or "None"
        )
        pdf.ln(2)
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(0, 6, "  Null Filling Actions:", new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", "", 9)
        for col, info in clean.get("nulls_filled",{}).items():
            pdf.cell(
                0, 5,
                clean_text(
                    f"    {col}: {info['count']} nulls filled"
                    f" using {info['strategy']} = {info['value']}"
                ),
                new_x="LMARGIN", new_y="NEXT"
            )
        pdf.ln(2)
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(0, 6, "  Outliers Flagged (IQR):", new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", "", 9)
        for col, info in clean.get("outliers_flagged",{}).items():
            pdf.cell(
                0, 5,
                clean_text(
                    f"    {col}: {info['count']} rows outside"
                    f" [{info['lower_bound']}, {info['upper_bound']}]"
                ),
                new_x="LMARGIN", new_y="NEXT"
            )
        pdf.ln(4)

        # Section 3 — Stats table
        pdf.section_header("3.  Statistical Summary")
        stats_df = df.describe().round(2).fillna("")
        n_cols   = len(stats_df.columns)
        col_w    = min(24, 165 / (n_cols + 1))

        pdf.set_font("Helvetica", "B", 8)
        pdf.set_fill_color(31, 97, 141)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(col_w, 6, "Stat", border=1, fill=True)
        for col in stats_df.columns:
            pdf.cell(col_w, 6, clean_text(str(col)[:9]), border=1, fill=True)
        pdf.ln()

        pdf.set_text_color(0, 0, 0)
        for i, idx in enumerate(stats_df.index):
            fill_color = (240, 247, 255) if i % 2 == 0 else (255, 255, 255)
            pdf.set_fill_color(*fill_color)
            pdf.set_font("Helvetica", "B", 8)
            pdf.cell(col_w, 5, clean_text(str(idx)), border=1, fill=True)
            pdf.set_font("Helvetica", "", 8)
            for col in stats_df.columns:
                pdf.cell(
                    col_w, 5,
                    safe_cell_value(stats_df.loc[idx, col]),
                    border=1, fill=True
                )
            pdf.ln()
        pdf.ln(4)

        # Section 4 — Correlations
        pdf.section_header("4.  Key Correlations")
        target = schema.get("target_variable","")
        corr   = eda.get("correlation",{})
        pdf.set_font("Helvetica", "", 10)
        if target and target in corr:
            target_corr = {
                k: v for k, v in corr[target].items() if k != target
            }
            sorted_corr = sorted(
                target_corr.items(),
                key=lambda x: abs(x[1]), reverse=True
            )
            for col, val in sorted_corr[:7]:
                try:
                    if pd.isna(val): continue
                except (TypeError, ValueError):
                    pass
                direction = "positive" if val > 0 else "negative"
                strength  = (
                    "strong"   if abs(val) > 0.4 else
                    "moderate" if abs(val) > 0.2 else "weak"
                )
                bar = "I" * int(abs(val) * 15)
                pdf.cell(
                    0, 6,
                    clean_text(
                        f"  {col:18} {val:+.4f}"
                        f"  ({strength} {direction})  {bar}"
                    ),
                    new_x="LMARGIN", new_y="NEXT"
                )
        pdf.ln(4)

        # Section 5 — AI Insights
        pdf.add_page()
        pdf.section_header(f"5.  AI Insights ({GEMINI_MODEL})")
        pdf.set_font("Helvetica", "", 10)
        pdf.multi_cell(0, 5, clean_text(insights))
        pdf.ln(4)

        # Section 6 — Categorical Summary
        pdf.section_header("6.  Categorical Column Summary")
        for col, counts in eda.get("value_counts",{}).items():
            pdf.set_font("Helvetica", "B", 10)
            pdf.cell(0, 7, clean_text(f"  {col}"), new_x="LMARGIN", new_y="NEXT")
            pdf.set_font("Helvetica", "", 9)
            for val, count in list(counts.items())[:6]:
                pct = round(count / eda["shape"][0] * 100, 1)
                bar = "I" * int(pct / 5)
                pdf.cell(
                    0, 5,
                    clean_text(f"    {str(val):18} : {count:5}  ({pct}%)  {bar}"),
                    new_x="LMARGIN", new_y="NEXT"
                )
            pdf.ln(2)

        # Section 7 — Pipeline Log
        pdf.add_page()
        pdf.section_header("7.  Pipeline Execution Log")
        pdf.set_font("Helvetica", "", 9)
        for entry in pipe_log:
            pdf.cell(
                0, 5, clean_text(f"  {entry}"),
                new_x="LMARGIN", new_y="NEXT"
            )

        if state["errors"]:
            pdf.ln(4)
            pdf.section_header("  Errors Encountered")
            pdf.set_font("Helvetica", "", 9)
            for err in state["errors"]:
                pdf.cell(
                    0, 5, clean_text(f"  ERROR: {err}"),
                    new_x="LMARGIN", new_y="NEXT"
                )

        # Save PDF — write directly to Volume (no /tmp on serverless)
        filename    = "insightforge_report.pdf"
        volume_path = f"/Volumes/insight/default/titanic/{filename}"

        pdf.output(volume_path)

        log = log_event(
            state, agent_name,
            f"done - PDF saved, {pdf.page_no()} pages"
        )

        logger.info(
            f"{agent_name} complete — "
            f"PDF saved to {volume_path}, "
            f"{pdf.page_no()} pages"
        )

        print(f"   Pages    : {pdf.page_no()}")
        print(f"   Saved to : {volume_path}")
        print(f"✅ {agent_name} complete")

        return {
            "pdf_path"     : volume_path,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "pdf_path"     : "",
            "pipeline_log" : log_event(
                state, agent_name, f"FAILED - {e}"
            ),
            "errors"       : errors + [msg]
        }

print("✅ report_agent() defined")
